## Given this code, what can you change in order to improve the performance here?

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Data
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# Model
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.Sigmoid(),
            nn.Linear(256, 128),
            nn.Sigmoid(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.layers(x)

# Train and val
def train_and_evaluate(epochs=10):
    model = SimpleNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total

        # Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_acc = 100 * correct / total
        print(f"Epoch {epoch+1} | Loss: {running_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

train_and_evaluate(epochs=10)

Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 34.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.29MB/s]


Epoch 1 | Loss: 1089.336 | Train Acc: 9.96% | Test Acc: 11.35%
Epoch 2 | Loss: 1079.621 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 3 | Loss: 1079.100 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 4 | Loss: 1078.956 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 5 | Loss: 1078.830 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 6 | Loss: 1078.693 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 7 | Loss: 1078.554 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 8 | Loss: 1078.437 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 9 | Loss: 1078.297 | Train Acc: 11.24% | Test Acc: 11.35%
Epoch 10 | Loss: 1078.170 | Train Acc: 11.24% | Test Acc: 11.35%
